# PneumoniaMNIST — Variational Autoencoder

Train and develop a **Variational Autoencoder** from scratch to generate 28×28 grayscale chest X-rays.
This notebook exports samples to the **Variational Autoencoder leaderboard**. VAE and diffusion entries have
independent best scores and ranks, and both boards appear together on the student/teacher screen.

Save a copy to Drive and select **Runtime → Change runtime type → T4 GPU** if available.
Your task is to develop the architecture and training recipe, record experiments, and discuss sample quality/diversity.
The starter trains one model; it does not automatically choose architectures or compare model families.

Dataset: [MedMNIST v2, Yang et al. (2023)](https://www.nature.com/articles/s41597-022-01721-8),
derived from [Kermany et al. (2018)](https://doi.org/10.1016/j.cell.2018.02.010).
PneumoniaMNIST uses CC BY 4.0. This exercise uses its standard 28×28 grayscale images.

In [ ]:
%pip install -q medmnist==3.0.2

In [ ]:
import hashlib
import importlib
from pathlib import Path
import random
import urllib.request

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import PneumoniaMNIST
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Path("data").mkdir(exist_ok=True)
transform = transforms.ToTensor()  # one grayscale channel, values in [0, 1]
train_data = PneumoniaMNIST(split="train", root="data", download=True, size=28, transform=transform)
validation_data = PneumoniaMNIST(split="val", root="data", download=False, size=28, transform=transform)
BATCH_SIZE = 128
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
validation_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print("Device:", device, "| Training images:", len(train_data), "| Validation:", len(validation_data))

## Generation benchmark

The server ranks a fixed-size batch of **256 new unconditional samples** by **RBF feature MMD²** against
a fixed held-out reference distribution. The frozen PneumoniaMNIST feature encoder, training-feature
normalisation and kernel bandwidth are identical for both boards. Lower scores are better.
This is a domain-feature distribution-matching benchmark, **not Inception-based FID or KID**.

The V-statistic includes diagonal terms, so it is non-negative apart from numerical rounding.
Also inspect feature diversity and image galleries: a score alone does not establish clinical realism.
Train only on training images and use validation losses to guide development. Do not copy images,
submit reconstructions, use pretrained models, or hand-select favourable generated samples.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/jamie-duell/interactive_coding-main/main/examples/data"
SOURCE_URL = "https://raw.githubusercontent.com/jamie-duell/interactive_coding-main/main/backend/app/challenges/generative_models"
for name, url in [
    ("encoder_model.npz", f"{DATA_URL}/generative_encoder_model.npz"),
    ("reference_distribution.npz", f"{DATA_URL}/generative_reference_distribution.npz"),
    ("generation_reference.py", f"{SOURCE_URL}/reference.py"),
]:
    urllib.request.urlretrieve(url, name)
import generation_reference
importlib.reload(generation_reference)
BENCHMARK_SHA256 = generation_reference.benchmark_sha256()
assert BENCHMARK_SHA256 == "1c0fcefb3ee2ac7a4d11168130c7466e40268f670c6cecd4dc963e39b66f637e", "Use the current notebook and benchmark together."

## Your VAE architecture — edit and experiment

The encoder produces a latent mean and log variance. Training uses the reparameterisation trick;
the decoder returns logits for 784 pixels. Explore hidden dimensions, latent dimension,
convolutional architectures and the reconstruction/KL trade-off. Keep the `VAE` interface below.
After architecture edits, rerun model creation and training.

In [ ]:
LATENT_DIM = 16


class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU())
        self.mu = nn.Linear(128, latent_dim)
        self.logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 128), nn.ReLU(), nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, 784))

    def forward(self, images):
        encoded = self.encoder(images)
        mu, logvar = self.mu(encoded), self.logvar(encoded).clamp(-10, 10)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return self.decoder(z), mu, logvar

    def generate(self, n, generator):
        z = torch.randn(n, self.latent_dim, device=device, generator=generator)
        return self.decoder(z).sigmoid().reshape(-1, 1, 28, 28)


model = VAE().to(device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## Train: reconstruction plus KL divergence

The reconstruction term is binary cross-entropy summed over pixels; KL divergence is summed
over latent dimensions. Both are averaged over images. `BETA` weights the KL term.
Use validation loss to choose the checkpoint within this run; compare final generation quality
and diversity when discussing experiments. Training losses are not the leaderboard metric.

In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-3
BETA = 1.0
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
best_state, best_validation = None, float("inf")
history = []

def vae_loss(images):
    logits, mu, logvar = model(images)
    reconstruction = F.binary_cross_entropy_with_logits(logits, images.flatten(1), reduction="none").sum(1)
    kl = -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(1)
    return (reconstruction + BETA * kl).mean()

for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for images, _ in train_loader:
        images = images.to(device)
        optimizer.zero_grad()
        loss = vae_loss(images)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(images)
    model.eval()
    validation = 0.0
    # Repeat the validation reparameterisation noise for comparable checkpoints.
    with torch.random.fork_rng(devices=[device.index or 0] if device.type == "cuda" else []):
        torch.manual_seed(123)
        with torch.inference_mode():
            for images, _ in validation_loader:
                validation += vae_loss(images.to(device)).item() * len(images)
    validation /= len(validation_data)
    history.append({"epoch": epoch + 1, "training_loss": total / len(train_data), "validation_loss": validation})
    if validation < best_validation:
        best_validation = validation
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    print(f"Epoch {epoch+1}: train {total/len(train_data):.3f}, validation {validation:.3f}")

assert best_state is not None, "Train for at least one epoch."
model.load_state_dict(best_state)
model.eval()

## Record your experiments

| Architecture / settings | Epochs | Validation loss | Feature MMD² | Feature diversity | What did you learn? |
|---|---:|---:|---:|---:|---|
| Starter | | | | | |
| Your change | | | | | |
| Your final choice | | | | | |

Discuss quality, diversity, training cost and generation speed. If you try both families,
your arena page keeps separate best results on the two leaderboards.

## Generate 256 new samples and evaluate

Use the fixed seed, export the first 256 generated images, and inspect the first 16.
Do not select favourable outputs or copy/reconstruct training, validation or test images.
The server recomputes this same quality score from your uploaded sample batch.

In [ ]:
SAMPLING_SEED = 2026  # fixed sampling seed for the exercise
generator = torch.Generator(device=device).manual_seed(SAMPLING_SEED)
with torch.inference_mode():
    generated = model.generate(256, generator).clamp(0, 1)
samples = generated[:, 0].cpu().numpy().astype(np.float32)
assert samples.shape == (256, 28, 28) and np.isfinite(samples).all()
quality, diversity = generation_reference.score(samples)
print(f"Feature MMD²: {quality:.4f} (lower is better); feature diversity: {diversity:.3f}")
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for ax, image in zip(axes.flat, samples[:16]):
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Export your experiment

Keep the model family below correct; it determines which leaderboard receives the submission.
Give your architecture/settings a short description. The server accepts samples only and cannot
verify the architecture from images, so keep the notebook as your experiment record.

In [ ]:
MODEL_FAMILY = "vae"
DESCRIPTION = f"Variational Autoencoder starter; {EPOCHS} epochs"  # update to match your experiment
assert 1 <= len(DESCRIPTION) <= 128
np.savez_compressed(f"{MODEL_FAMILY}_samples.npz", samples=samples.astype(np.float32),
                    model_family=np.array(MODEL_FAMILY), benchmark_sha256=np.array(BENCHMARK_SHA256),
                    description=np.array(DESCRIPTION))
print(f"Upload {MODEL_FAMILY}_samples.npz on your arena student page.")
try:
    from google.colab import files
    files.download(f"{MODEL_FAMILY}_samples.npz")
except ImportError:
    print("Outside Colab: use the notebook file browser.")